# Week 3 — The Asset Management Game (slide 57)

Rules, straight from the deck:
- A mini stock market of 8 EGX names.
- Player must always hold exactly 2 stocks at a time (50/50 split).
- Playtime: 06 Jul – 04 Aug.
- Goal: maximize profit.

**Two things the slide leaves open, resolved here explicitly:**

1. **The 8 names.** Slide 57 only names three tickers directly, as example weekly
   picks: `QALA`, `BTFH`, `PHDC`. `QALA` (QALA For Financial Investments) has no
   committed CSV in `data/egx/`, so it's dropped. The other 5 names were chosen
   by hand to round the mini market out to 8, from liquid EGX names already used
   elsewhere in the course (`COMI`, `SWDY`, `ETEL`, `HRHO`, `FWRY`). Edit
   `GAME_UNIVERSE` below if your version of the deck names different ones.
2. **The year.** "06 Jul – 04 Aug" has no year on the slide. The committed data
   for these 7 tickers ends 2026-07-30 — a few trading days short of a full
   2026-08-04 close — so this notebook plays the most recent *complete*
   occurrence of that window: **06 Jul – 04 Aug 2025**. Change `PLAYTIME_YEAR`
   below for a different year (if you have data that covers it).

Player picks a 2-stock pair at each weekly checkpoint, using only price history
available up to that point — no lookahead. Since this notebook has to run
unattended (`Run All`, no `input()` / widgets / prompts), the "player" is a
small hardcoded momentum rule: at each checkpoint, hold the 2 stocks with the
best trailing return.


In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tradinglab.data_feed import DataFeed
from tradinglab.simulator import PortfolioSimulator
from tradinglab.metrics import total_return
from tradinglab.charting import plot_equity

print('cwd:', os.getcwd())


## 1. The mini market — 8 names from the slide, 7 we can actually play

In [ ]:
GAME_UNIVERSE = ['QALA', 'BTFH', 'PHDC', 'COMI', 'SWDY', 'ETEL', 'HRHO', 'FWRY']
PLAYABLE = [s for s in GAME_UNIVERSE if s != 'QALA']

DATA_DIR = Path('data/egx')
for s in PLAYABLE:
    assert (DATA_DIR / f'{s}.csv').exists(), f'expected data/egx/{s}.csv to exist'
assert not (DATA_DIR / 'QALA.csv').exists(), \
    'QALA was expected to be missing from data/egx — update PLAYABLE if this changes'

print(f'8-stock game universe (slide 57): {GAME_UNIVERSE}')
print(f'playable ({len(PLAYABLE)}/8 — QALA has no committed data): {PLAYABLE}')


## 2. Load the feed and pin the playtime window

In [ ]:
feed = DataFeed.from_dir('data/egx', symbols=PLAYABLE)
print('feed universe:', feed.symbols)
print('feed date range:', feed.dates.min().date(), '->', feed.dates.max().date(),
      f'({feed.n_days} trading days)')

PLAYTIME_YEAR = 2025
PLAYTIME_START = pd.Timestamp(f'{PLAYTIME_YEAR}-07-06')
PLAYTIME_END = pd.Timestamp(f'{PLAYTIME_YEAR}-08-04')

start_idx = feed.dates.get_loc(PLAYTIME_START)
end_idx = feed.dates.get_loc(PLAYTIME_END)  # sim.run's `end` is exclusive of the decision loop but its return IS included

print(f'playtime: {PLAYTIME_START.date()} -> {PLAYTIME_END.date()}  '
      f'(day index {start_idx} -> {end_idx})')


## 3. Weekly checkpoints — pick a pair using only data seen so far

06 Jul, 13 Jul, 20 Jul, 27 Jul are the four Sunday checkpoints inside the
playtime window (EGX's trading week runs Sunday–Thursday). At each one, the
rule looks back `MOMENTUM_LOOKBACK` trading days and holds the 2 stocks with
the best trailing return, 50/50, until the next checkpoint — nothing past the
checkpoint's own closing price is ever touched.


In [ ]:
CHECKPOINT_DATES = pd.to_datetime([
    f'{PLAYTIME_YEAR}-07-06', f'{PLAYTIME_YEAR}-07-13',
    f'{PLAYTIME_YEAR}-07-20', f'{PLAYTIME_YEAR}-07-27',
])
checkpoint_idx = [feed.dates.get_loc(d) for d in CHECKPOINT_DATES]

MOMENTUM_LOOKBACK = 15  # trading days (~3 EGX weeks) of history used at each checkpoint


def pick_pair(feed, cp_idx, lookback):
    """Top-2 trailing-return stocks as of `cp_idx`'s close — nothing past cp_idx is touched."""
    past = feed.close[cp_idx - lookback]
    now = feed.close[cp_idx]
    momentum = now / past - 1.0
    ranked = np.argsort(momentum)[::-1]
    return ranked[0], ranked[1]


weights_by_day = np.zeros((feed.n_days, feed.n_assets))
weekly_picks = []
for week_i, cp_idx in enumerate(checkpoint_idx):
    i, j = pick_pair(feed, cp_idx, MOMENTUM_LOOKBACK)
    hold_until = checkpoint_idx[week_i + 1] if week_i + 1 < len(checkpoint_idx) else end_idx
    weights_by_day[cp_idx:hold_until, i] = 0.5
    weights_by_day[cp_idx:hold_until, j] = 0.5
    weekly_picks.append((CHECKPOINT_DATES[week_i].date(), feed.symbols[i], feed.symbols[j]))

assert np.allclose(weights_by_day[start_idx:end_idx].sum(axis=1), 1.0), \
    'must always hold exactly 2 stocks at 50/50'

for d, a, b in weekly_picks:
    print(f'  {d}  ->  hold {a} / {b}  (50/50)')


## 4. Run the game strategy through the simulator

In [ ]:
sim = PortfolioSimulator(feed)  # benchmark='equal_weight' over exactly these 7 stocks
result = sim.run(weights_by_day, start_idx, end_idx)

print('trading days played:', len(result['portfolio']))
print('final strategy value (start = 1.0): ', round(result['portfolio'][-1], 4))
print('final benchmark value (start = 1.0):', round(result['benchmark'][-1], 4))


## 5. Best- and worst-possible single pair, for context

Brute force every one of the C(7,2) = 21 possible pairs, each held flat
(50/50, no rebalancing) for the whole playtime window — through the SAME
simulator, so the comparison is apples-to-apples with the weekly strategy above.


In [ ]:
def flat_pair_result(feed, sim, i, j, start, end):
    w = np.zeros((feed.n_days, feed.n_assets))
    w[start:end, i] = 0.5
    w[start:end, j] = 0.5
    return sim.run(w, start, end)


pair_returns = {}
for i, j in itertools.combinations(range(feed.n_assets), 2):
    res = flat_pair_result(feed, sim, i, j, start_idx, end_idx)
    pair_returns[(feed.symbols[i], feed.symbols[j])] = total_return(res['portfolio_returns'])

best_pair, best_ret = max(pair_returns.items(), key=lambda kv: kv[1])
worst_pair, worst_ret = min(pair_returns.items(), key=lambda kv: kv[1])

print(f'best possible flat pair over the playtime window:  '
      f'{best_pair[0]}/{best_pair[1]}  {best_ret:+.1%}')
print(f'worst possible flat pair over the playtime window: '
      f'{worst_pair[0]}/{worst_pair[1]}  {worst_ret:+.1%}')


## 6. Scorecard — weekly game strategy vs equal-weight benchmark

In [ ]:
game_return = total_return(result['portfolio_returns'])
benchmark_return = total_return(result['benchmark_returns'])

print(f'{"weekly game strategy":34s} {game_return:+.1%}')
print(f'{"equal-weight benchmark (7 stocks)":34s} {benchmark_return:+.1%}')
print(f'{"best possible flat pair":34s} {best_ret:+.1%}   ({best_pair[0]}/{best_pair[1]})')
print(f'{"worst possible flat pair":34s} {worst_ret:+.1%}   ({worst_pair[0]}/{worst_pair[1]})')

plot_equity(
    result['portfolio'], result['benchmark'], dates=result['dates'],
    title=f'Asset Management Game — weekly picks vs equal-weight benchmark '
          f'({PLAYTIME_START.date()} to {PLAYTIME_END.date()})',
)
plt.show()
